In [ ]:
# DO NOT RUN ANYMORE!!!!
# WE SAVE EVERYTHING IN SYNTHETIC/

EPOCHS     = 100
BATCH_SIZE = 16

n_runs = 5   # TO CHANGE
n_iter = 100



def make_adam_tracker(loss_list, weights_list=None, maxiter=100, N=4):
    log_n = np.log2(N)
    def adam_with_tracking(fun, x0, jac=None, **kwargs):
        optimizer = ADAM(maxiter=maxiter, lr=0.1, beta_1=0.9, beta_2=0.99, tol=1e-8)
        _cache = {'weights': None, 'loss': None}
        def cached_fun(weights):
            if _cache['weights'] is None or not np.array_equal(weights, _cache['weights']):
                _cache['weights'] = weights.copy()
                _cache['loss']    = fun(weights)
            return _cache['loss']
        def tracked_jac(weights):
            loss = cached_fun(weights)
            loss_list.append(float(loss) / log_n)
            if weights_list is not None:
                weights_list.append(weights.copy())
            return jac(weights)
        return optimizer.minimize(fun=cached_fun, x0=x0, jac=tracked_jac)
    return adam_with_tracking


def run_synthetic(seed):
    algorithm_globals.random_seed = seed
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)

    q_loss  = []
    eq_loss = []
    q_weights  = []   
    eq_weights = []

    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=make_adam_tracker(q_loss, q_weights),
        sampler=StatevectorSampler(),
    )
    e_vqc = VQC(
        feature_map=easy_feature_map,
        ansatz=ansatz,
        optimizer=make_adam_tracker(eq_loss, eq_weights),
        sampler=StatevectorSampler(),
    )
    CL_model = build_model2(input_lenght=4, depth=5, n_class=2)

    vqc.fit(X_s_tr, y_s_tr)
    e_vqc.fit(X_s_tr, y_s_tr)
    history = CL_model.fit(
        X_s_tr, y_s_tr,
        validation_data=(X_s_test, y_s_test),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        verbose=0, shuffle=True
    )

    q_acc = []
    for w in q_weights:
        preds = np.argmax(vqc.neural_network.forward(X_s_test, w), axis=1)
        q_acc.append(accuracy_score(y_s_test, preds))

    eq_acc = []
    for w in eq_weights:
        preds = np.argmax(e_vqc.neural_network.forward(X_s_test, w), axis=1)
        eq_acc.append(accuracy_score(y_s_test, preds))

    cl_loss = np.array(history.history['loss'])
    cl_acc  = np.array(history.history['val_accuracy'])

    return cl_loss, np.array(q_loss), np.array(eq_loss), cl_acc, np.array(q_acc), np.array(eq_acc)


for seed in range(n_runs):
    print(f'Running seed {seed}')
    cll, ql, eql, clacc, qacc, eqacc = run_synthetic(seed)
    #cl_all_losses.append(cll)
    #cl_all_accs.append(clacc)
    #q_all_losses.append(ql)
    #q_all_accs.append(qacc)
    #eq_all_losses.append(eql)
    #eq_all_accs.append(eqacc)

    with open(f"SYNTHETIC/cll_{seed}.pkl",  "wb") as f: pickle.dump(cll,  f)
    with open(f"SYNTHETIC/ql_{seed}.pkl", "wb") as f: pickle.dump(ql,   f)
    with open(f"SYNTHETIC/eql_{seed}.pkl",  "wb") as f: pickle.dump(eql,    f)
    with open(f"SYNTHETIC/clacc_{seed}.pkl",  "wb") as f: pickle.dump(clacc,    f)
    with open(f"SYNTHETIC/qacc_{seed}.pkl",  "wb") as f: pickle.dump(qacc,    f)
    with open(f"SYNTHETIC/equacc_{seed}.pkl",  "wb") as f: pickle.dump(eqacc,    f)

    # Non accumula nulla in RAM — scarta i risultati
    del cll, ql, eql, clacc, qacc, eqacc
    gc.collect()

In [ ]:
# LOAD SYNTHETIC RUNS
EPOCHS     = 100
BATCH_SIZE = 16

n_runs = 5   # TO CHANGE
n_iter = 100

cl_all_losses = []
cl_all_accs   = []
q_all_losses  = []
q_all_accs    = []
eq_all_losses = []
eq_all_accs   = []

for seed in range(n_runs):
    with open(f"SYNTHETIC/cll_{seed}.pkl",  "rb") as f:    cl_all_losses.append(pickle.load(f))
    with open(f"SYNTHETIC/ql_{seed}.pkl", "rb") as f:      q_all_losses.append(pickle.load(f))
    with open(f"SYNTHETIC/eql_{seed}.pkl",  "rb") as f:    eq_all_losses.append(pickle.load(f))
    with open(f"SYNTHETIC/clacc_{seed}.pkl",  "rb") as f:  cl_all_accs.append(pickle.load(f))
    with open(f"SYNTHETIC/qacc_{seed}.pkl",  "rb") as f:   q_all_accs .append(pickle.load(f))
    with open(f"SYNTHETIC/equacc_{seed}.pkl",  "rb") as f: eq_all_accs.append(pickle.load(f))


cl_all_losses = np.array(cl_all_losses) # (n_run, n_iter)
cl_all_accs = np.array(cl_all_accs)
q_all_losses = np.array(q_all_losses)
q_all_accs = np.array(q_all_accs)
eq_all_losses = np.array(eq_all_losses)
eq_all_accs = np.array(eq_all_accs)


# mean and std
cl_loss_mean = cl_all_losses.mean(axis=0).ravel()
cl_loss_std = cl_all_losses.std(axis=0).ravel()
cl_accs_mean = cl_all_accs.mean(axis=0).ravel()
cl_accs_std = cl_all_accs.std(axis=0).ravel()

q_loss_mean = q_all_losses.mean(axis=0).ravel()
q_loss_std = q_all_losses.std(axis=0).ravel()
q_accs_mean = q_all_accs.mean(axis=0).ravel()
q_accs_std = q_all_accs.std(axis=0).ravel()

eq_loss_mean = eq_all_losses.mean(axis=0).ravel()
eq_loss_std = eq_all_losses.std(axis=0).ravel()
eq_accs_mean = eq_all_accs.mean(axis=0).ravel()
eq_accs_std = eq_all_accs.std(axis=0).ravel()


In [ ]:
# plot loss
plt.figure(figsize=(12,5))

plt.plot(cl_loss_mean, label="Mean CL loss", c='r')
plt.fill_between(
    range(n_iter),
    cl_loss_mean - cl_loss_std,
    cl_loss_mean + cl_loss_std,
    color='r',
    alpha=0.3
)

plt.plot(q_loss_mean, label="Mean QNN loss", c='b')
plt.fill_between(
    range(n_iter),
    q_loss_mean - q_loss_std,
    q_loss_mean + q_loss_std,
    color='b',
    alpha=0.3
)

plt.plot(eq_loss_mean, label="Mean EASY QNN loss", c='g')
plt.fill_between(
    range(n_iter),
    eq_loss_mean - eq_loss_std,
    eq_loss_mean + eq_loss_std,
    color='g',
    alpha=0.3
)

plt.title("Training Loss over synthetic dataset")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid()
plt.legend()
plt.show()


# plot accuracy
plt.figure(figsize=(12,5))

plt.plot(cl_accs_mean, label="Mean CL accuracy", c='r')
plt.fill_between(
    range(n_iter),
    cl_accs_mean - cl_accs_std,
    cl_accs_mean + cl_accs_std,
    color='r',
    alpha=0.3
)

plt.plot(q_accs_mean, label="Mean QNN accuracy", c='b')
plt.fill_between(
    range(n_iter),
    q_accs_mean - q_accs_std,
    q_accs_mean + q_accs_std,
    color='b',
    alpha=0.3
)

plt.plot(eq_accs_mean, label="Mean EASY QNN accuracy", c='g')
plt.fill_between(
    range(n_iter),
    eq_accs_mean - eq_accs_std,
    eq_accs_mean + eq_accs_std,
    color='g',
    alpha=0.3
)

plt.title("Test Accuracy over synthetic dataset")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# DO NOT RUN ANYMORE!!!
# WE SAVE EVERYTHING IN BINARY/
n_runs = 5   # TO CHANGE
n_iter = 100



def run_binary(seed):
    algorithm_globals.random_seed = seed
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)

    q_loss  = []
    eq_loss = []
    q_weights  = []
    eq_weights = []

    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=make_adam_tracker(q_loss, q_weights),
        sampler=StatevectorSampler(),
    )
    e_vqc = VQC(
        feature_map=easy_feature_map,
        ansatz=ansatz,
        optimizer=make_adam_tracker(eq_loss, eq_weights),
        sampler=StatevectorSampler(),
    )
    CL_model = build_model2(input_lenght=4, depth=5, n_class=2)

    vqc.fit(X_b_tr, y_b_tr)
    e_vqc.fit(X_b_tr, y_b_tr)
    history = CL_model.fit(
        X_b_tr, y_b_tr,
        validation_data=(X_b_test, y_b_test),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        verbose=0, shuffle=True
    )

    q_acc = []
    for w in q_weights:
        preds = np.argmax(vqc.neural_network.forward(X_b_test, w), axis=1)
        q_acc.append(accuracy_score(y_b_test, preds))

    eq_acc = []
    for w in eq_weights:
        preds = np.argmax(e_vqc.neural_network.forward(X_b_test, w), axis=1)
        eq_acc.append(accuracy_score(y_b_test, preds))

    cl_loss = np.array(history.history['loss'])
    cl_acc  = np.array(history.history['val_accuracy'])

    return cl_loss, np.array(q_loss), np.array(eq_loss), cl_acc, np.array(q_acc), np.array(eq_acc)


for seed in range(n_runs):
    print(f'Running seed {seed}')
    cll, ql, eql, clacc, qacc, eqacc = run_binary(seed)
    cl_all_losses.append(cll)
    cl_all_accs.append(clacc)
    q_all_losses.append(ql)
    q_all_accs.append(qacc)
    eq_all_losses.append(eql)
    eq_all_accs.append(eqacc)
    with open(f"BINARY/cll_{seed}.pkl",  "wb") as f: pickle.dump(cll,  f)
    with open(f"BINARY/ql_{seed}.pkl", "wb") as f: pickle.dump(ql,   f)
    with open(f"BINARY/eql_{seed}.pkl",  "wb") as f: pickle.dump(eql,    f)
    with open(f"BINARY/clacc_{seed}.pkl",  "wb") as f: pickle.dump(clacc,    f)
    with open(f"BINARY/qacc_{seed}.pkl",  "wb") as f: pickle.dump(qacc,    f)
    with open(f"BINARY/equacc_{seed}.pkl",  "wb") as f: pickle.dump(eqacc,    f)

In [ ]:
# LOAD BINARY RUNS
cl_all_losses = []
cl_all_accs   = []
q_all_losses  = []
q_all_accs    = []
eq_all_losses = []
eq_all_accs   = []

for seed in range(n_runs):
    with open(f"BINARY/cll_{seed}.pkl",  "rb") as f:    cl_all_losses.append(pickle.load(f))
    with open(f"BINARY/ql_{seed}.pkl", "rb") as f:      q_all_losses.append(pickle.load(f))
    with open(f"BINARY/eql_{seed}.pkl",  "rb") as f:    eq_all_losses.append(pickle.load(f))
    with open(f"BINARY/clacc_{seed}.pkl",  "rb") as f:  cl_all_accs.append(pickle.load(f))
    with open(f"BINARY/qacc_{seed}.pkl",  "rb") as f:   q_all_accs .append(pickle.load(f))
    with open(f"BINARY/equacc_{seed}.pkl",  "rb") as f: eq_all_accs.append(pickle.load(f))

cl_all_losses = np.array(cl_all_losses) # (n_run, n_iter)
cl_all_accs = np.array(cl_all_accs)
q_all_losses = np.array(q_all_losses)
q_all_accs = np.array(q_all_accs)
eq_all_losses = np.array(eq_all_losses)
eq_all_accs = np.array(eq_all_accs)


# mean and std
cl_loss_mean = cl_all_losses.mean(axis=0).ravel()
cl_loss_std = cl_all_losses.std(axis=0).ravel()
cl_accs_mean = cl_all_accs.mean(axis=0).ravel()
cl_accs_std = cl_all_accs.std(axis=0).ravel()

q_loss_mean = q_all_losses.mean(axis=0).ravel()
q_loss_std = q_all_losses.std(axis=0).ravel()
q_accs_mean = q_all_accs.mean(axis=0).ravel()
q_accs_std = q_all_accs.std(axis=0).ravel()

eq_loss_mean = eq_all_losses.mean(axis=0).ravel()
eq_loss_std = eq_all_losses.std(axis=0).ravel()
eq_accs_mean = eq_all_accs.mean(axis=0).ravel()
eq_accs_std = eq_all_accs.std(axis=0).ravel()


In [ ]:
# plot loss
plt.figure(figsize=(12,5))

plt.plot(cl_loss_mean, label="Mean CL loss", c='r')
plt.fill_between(
    range(n_iter),
    cl_loss_mean - cl_loss_std,
    cl_loss_mean + cl_loss_std,
    color='r',
    alpha=0.3
)

plt.plot(q_loss_mean, label="Mean QNN loss", c='b')
plt.fill_between(
    range(n_iter),
    q_loss_mean - q_loss_std,
    q_loss_mean + q_loss_std,
    color='b',
    alpha=0.3
)

plt.plot(eq_loss_mean, label="Mean EASY QNN loss", c='g')
plt.fill_between(
    range(n_iter),
    eq_loss_mean - eq_loss_std,
    eq_loss_mean + eq_loss_std,
    color='g',
    alpha=0.3
)

plt.title("Training Loss over the binary dataset ")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid()
plt.legend()
plt.show()


# plot accuracy
plt.figure(figsize=(12,5))

plt.plot(cl_accs_mean, label="Mean CL accuracy", c='r')
plt.fill_between(
    range(n_iter),
    cl_accs_mean - cl_accs_std,
    cl_accs_mean + cl_accs_std,
    color='r',
    alpha=0.3
)

plt.plot(q_accs_mean, label="Mean QNN accuracy", c='b')
plt.fill_between(
    range(n_iter),
    q_accs_mean - q_accs_std,
    q_accs_mean + q_accs_std,
    color='b',
    alpha=0.3
)

plt.plot(eq_accs_mean, label="Mean EASY QNN accuracy", c='g')
plt.fill_between(
    range(n_iter),
    eq_accs_mean - eq_accs_std,
    eq_accs_mean + eq_accs_std,
    color='g',
    alpha=0.3
)

plt.title("Test Accuracy over the binary dataset")
plt.xlabel("Iteration")
plt.ylabel("Accuracy")
plt.grid()
plt.legend()
plt.show()